## ⛏️ Data Flows Extraction with Redis

Data Flows Extraction relying on a Redis server.

In [1]:
# Imports
from   dotenv   import load_dotenv
import pandas   as pd
import datetime
import redis
import json
import sys
import os

# Add the upper folder to sys.path
sys.path.insert(0, "../../")
from   RedisClient import RedisClient
from   App         import App
from   App         import DataFlows
import Utils

#### Parameters

In [2]:
# TMP Folder
TMP_PATH = "../../../../0_Data/TMP/"

#### Initialization

In [3]:
print("⚡ Start - {} ⚡\n".format(datetime.datetime.now()))
startTime = datetime.datetime.now()

⚡ Start - 2024-07-28 12:46:09.626347 ⚡



In [4]:
# Create TMP Folder
if not os.path.exists(TMP_PATH):
	os.makedirs(TMP_PATH)
	print("📁🆕 Folder created       :", TMP_PATH)
else:
	print("📁✅ Folder already exists:", TMP_PATH)

📁✅ Folder already exists: ../../../../0_Data/TMP/


#### 📥 1) Read Data and Push to Redis

In [5]:
# Datasets
#DATASET = "0_AndroCatSet"
#DATASET = "1_AndroCatSetMini"
#DATASET = "2_AndroCatSetTestSet"
DATASET = "3_MalCatSet"

# Direction
#DIRECTION	= "backward"
DIRECTION	= "forward"

# List of source and sinks
#SOURCES_APPROACH = "marco"
#SOURCES_APPROACH = "susi"
SOURCES_APPROACH  = "docflow"

# Only pairs or full paths
FULL_PATHS 		= "false"
# FULL_PATHS     = "true"

In [6]:
if DATASET in ["0_AndroCatSet", "1_AndroCatSetMini", "2_AndroCatSetTestSet"]: 
	REDIS_PROJECT_KEY = "marco.dataflow.extraction.androcatset"
	
if DATASET == "3_MalCatSet": 
	REDIS_PROJECT_KEY = "marco.dataflow.extraction.malcatset"

In [7]:
if DIRECTION == "backward":
    REDIS_PROJECT_KEY += ".backward.goldensinks.pairs"
elif DIRECTION == "forward":
    if SOURCES_APPROACH == "susi":
        REDIS_PROJECT_KEY += ".forward.susi.pairs"
    elif SOURCES_APPROACH == "docflow":
        REDIS_PROJECT_KEY += ".forward.docflow.pairs"

In [8]:
print("--- 🔑 Redis Key: ", REDIS_PROJECT_KEY)

--- 🔑 Redis Key:  marco.dataflow.extraction.malcatset.forward.docflow.pairs


📡 Redis connection

In [9]:
# Connection to REDIS
load_dotenv()
redisClientExtraction = RedisClient(host="serval06.uni.lux", port=6379, db=0, password=os.getenv("REDIS_PSW"), projectKey = REDIS_PROJECT_KEY)

📜 Print Size of Lists/Sets

In [10]:
redisClientExtraction.printStatus()

⏲️ Current Date and Time: 2024-07-28 12:46:09.658185

📐 Size
- marco.dataflow.extraction.malcatset.forward.docflow.pairs.pop : 2161
- marco.dataflow.extraction.malcatset.forward.docflow.pairs.result : 1081
- marco.dataflow.extraction.malcatset.forward.docflow.pairs.error : 315


In [35]:
# Path
DATA_PATH = "../../../../0_Data/{}.csv".format(DATASET) 

# Read the data
appsDF = pd.read_csv(DATA_PATH)

# Print Number
print("--- #️⃣ Apps: {} ".format(appsDF.shape[0]))

# # # Removing apps wihtout a class
#appsDF = appsDF.dropna()
#print("--- #️⃣ Removing Nan Values : {}".format(appsDF.shape[0]))

# TEST
#appsDF = appsDF.head(10)
appsDF.head(5)

--- #️⃣ Apps: 1805 


,sha256,pkgName,classID
0,24D3490CF23842A791CBB5B10F1427808F4B163F9C4927...,com.rytong.airchina,Airlines
1,2D3D869A1DF82ACDCABBB08277ADECB8E5B64DB4DB516C...,com.rytong.hnair,Airlines
2,A1B554693AD6B506B179AE6AEC858205B011073E2A58A7...,com.hkairlines.apps,Airlines
3,2EA3916D3E63B3FE084B68AD5353A88C6C0F2A62A0B9BC...,com.fullchargepro.batteryalarmsignal,Alarm
4,2E91A25E7D30DCE7BA8217EBA79EABA230C445E56367E3...,com.splunchy.android.alarmclock,Alarm


In [36]:
# #To push
# redisClientExtraction.loadPopList(list(appsDF['sha256'].values))

#### 🔁 Extraction Loop Execution: Pop from Redis and extract, then push results back.

In [87]:
# Path to Android Platforms
ANDROID_PATH     = "/home/marco/android/platforms"
# ANDROID_PATH     = "/home/users/malecci/android/platforms"

# Path to the Java Script used for Data Flows Extaction
JAVA_EXTRACTOR_PATH = "../../../Java/dataflowextractor/target/dataflowextractor-1.0-SNAPSHOT-jar-with-dependencies.jar"

# Timeout for Data Flow Analysis
TIMEOUT = 300

In [88]:
# Pop from Redis popList
while (sha256 := redisClientExtraction.client.rpop(redisClientExtraction.popKey) ) is not None:

	# Get sha256
	sha256 = sha256.decode("utf-8") 
	print("--------------------------------------------")
	print("🔑 Analyzing APK: {}".format(sha256))

	  # Skip if already processed
	if redisClientExtraction.client.hget(redisClientExtraction.resultsKey, sha256) is not None:
		print("\n⏭️  Already Processed --> Skip")
		continue
   
	# Launch Difuzer
	try:
		# Create App instance
		app = App(sha256 = sha256)

		# Extract data flows
		app.extractDataFlows(TMP_PATH, JAVA_EXTRACTOR_PATH, ANDROID_PATH, DIRECTION, SOURCES_APPROACH, FULL_PATHS, TIMEOUT)

		# Convert to JSON
		jsonString = app.dataFlows.toJsonString()

		# Store results into Redis
		redisClientExtraction.client.hset(redisClientExtraction.resultsKey, sha256, jsonString)
		print("\n✅ Success for APK: {}".format(sha256), flush=True)
		
	# Print exception and store into errorList
	except Exception as e:
		print("\n❌  Failed with Exception {}".format(e), flush=True)
		redisClientExtraction.client.lpush(redisClientExtraction.errorKey, sha256)
	print("--------------------------------------------\n\n")

--------------------------------------------
🔑 Analyzing APK: 01F0AEDB6078A0D4CC46ACCC0730EA73166D979F05BF056C48B946C2ED9A0DFB
--- 📥 Downloading APK
--- 🔄 Tentive N: 0
--- ❌ Error: Received status code 503. Retrying in 30 seconds...
--- 🔄 Tentive N: 1
--- ❌ Error: Received status code 503. Retrying in 30 seconds...
--- 🔄 Tentive N: 2
--- 📤 APK file downloaded and saved to ../../../../0_Data/TMP/01F0AEDB6078A0D4CC46ACCC0730EA73166D979F05BF056C48B946C2ED9A0DFB.apk
--- 💻 Executing: java -Xmx24g -Xss1g -jar ../../../Java/dataflowextractor/target/dataflowextractor-1.0-SNAPSHOT-jar-with-dependencies.jar -a ../../../../0_Data/TMP/01F0AEDB6078A0D4CC46ACCC0730EA73166D979F05BF056C48B946C2ED9A0DFB.apk -p /home/marco/android/platforms -s true -d backward -sources marco -fullPaths false
--- ⏲️ Timeout: 300 s


+++ START of Logging/Error +++
Exception in thread "main" soot.AndroidPlatformException: Android platform directory '/home/marco/android/platforms' does not exist!
	at soot.Scene.getAndroidAP